# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_04 — Return Sanitization and Bias Control

### Research question

How can we detect, flag, and control common market-data problems before returns enter alpha research, portfolio construction, or backtesting?

This notebook focuses on four major sources of fake or distorted performance:

1. **Outliers and bad ticks**
2. **Corporate actions and adjusted prices**
3. **Survivorship bias**
4. **Missingness and stale observations**

The goal is not to make data look clean by blindly deleting inconvenient observations. The goal is to produce an auditable return panel where each correction, exclusion, and warning is explicit.

A useful rule for quant research is:

> If a return enters a model, the pipeline should be able to explain where it came from and whether it was adjusted, imputed, flagged, or excluded.

## 1. Why return sanitization matters

Price data can look harmless while return data is broken.

A single corrupted price can create two false returns:

1. an artificial jump into the bad price;
2. an artificial reversal out of the bad price.

Corporate actions can create similar patterns if raw prices are used instead of adjusted prices.

Survivorship bias can make strategies look better by excluding failed or delisted assets.

Missing data can create hidden look-ahead bias if gaps are filled incorrectly.

This notebook therefore separates the workflow into five layers:

| Layer | Purpose |
|---|---|
| Raw panel | Prices as received from the vendor |
| Diagnostic flags | Detect suspicious observations |
| Corporate action handling | Distinguish economic moves from mechanical price changes |
| Bias-aware universe | Include assets that existed at the time, not only final survivors |
| Sanitized returns | Return panel with explicit quality metadata |

## 2. Key concepts

### 2.1 Raw return

For raw closing prices $P_t$, the simple return is:

$$
r_t = \frac{P_t}{P_{t-1}} - 1
$$
or the log return is:

$$
\ell_t = \log(P_t) - \log(P_{t-1})
$$
Raw prices can be distorted by stock splits, reverse splits, and other corporate actions.

### 2.2 Adjusted close

Adjusted close attempts to restate historical prices so that returns reflect the economic value of holding the asset rather than mechanical changes in share count or distributions.

In this notebook, we simulate split-adjusted prices using:

$$
P_t^{\text{adjusted}} = P_t^{\text{raw}} \times A_t
$$
where $A_t$ is the cumulative split adjustment factor.

### 2.3 Survivorship bias

Survivorship bias occurs when the universe used in a backtest contains only assets that survived until the end of the sample.

This can overstate returns and understate risk because failed, delisted, merged, or inactive assets are removed from history.

### 2.4 Missingness

Missing data can be:

- random and harmless;
- caused by holidays or trading suspensions;
- caused by illiquidity;
- caused by delisting;
- caused by vendor outages;
- caused by corporate actions or identifier changes.

The reason for missingness matters.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## 3. Synthetic equity universe with controlled data defects

To test the sanitization pipeline, we generate a synthetic equity panel with known defects.

The synthetic dataset includes:

1. normal price paths;
2. stock splits;
3. reverse splits;
4. delistings;
5. random missing values;
6. block missing values;
7. injected bad ticks / outliers.

Because the defects are known, we can evaluate whether the diagnostic pipeline finds them.

In [ ]:
@dataclass(frozen=True)
class SyntheticEquityConfig:
    """
    Configuration for the synthetic equity panel.
    """
    start: str = "2020-01-01"
    end: str = "2024-12-31"
    num_assets: int = 80
    annual_market_vol: float = 0.22
    annual_idio_vol_low: float = 0.18
    annual_idio_vol_high: float = 0.45
    split_probability: float = 0.18
    reverse_split_probability: float = 0.07
    delist_probability: float = 0.20
    random_missing_probability: float = 0.006
    block_missing_probability: float = 0.18
    outlier_probability: float = 0.12
    seed: int = 42

In [ ]:
def simulate_clean_economic_prices(config: SyntheticEquityConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Simulate clean economic adjusted prices before injecting vendor defects.

    Returns
    -------
    clean_panel:
        Long panel containing clean economic prices.

    asset_metadata:
        Metadata including quality score and future delisting plan.
    """
    local_rng = np.random.default_rng(config.seed)

    dates = pd.date_range(config.start, config.end, freq="B", tz="UTC")
    n_dates = len(dates)
    symbols = [f"EQ{i:03d}" for i in range(1, config.num_assets + 1)]

    dt = 1 / 252

    market_returns = (
        0.06 * dt
        + config.annual_market_vol * np.sqrt(dt) * local_rng.standard_normal(n_dates)
    )

    metadata_rows = []
    frames = []

    for symbol in symbols:
        quality = local_rng.normal(loc=0.0, scale=1.0)

        beta = np.clip(local_rng.normal(loc=1.0, scale=0.25), 0.3, 1.8)
        annual_alpha = 0.03 * quality - 0.015
        idio_vol = local_rng.uniform(config.annual_idio_vol_low, config.annual_idio_vol_high)

        idio_returns = (
            annual_alpha * dt
            + idio_vol * np.sqrt(dt) * local_rng.standard_normal(n_dates)
        )

        log_returns = beta * market_returns + idio_returns

        s0 = local_rng.uniform(20, 150)
        clean_adj_close = s0 * np.exp(np.cumsum(log_returns))

        # Delisting is more likely for low-quality assets.
        delist_score = local_rng.uniform()
        delisted = delist_score < config.delist_probability * (1.5 if quality < 0 else 0.7)

        delist_date = pd.NaT
        delist_shock = 0.0

        if delisted:
            earliest = int(0.35 * n_dates)
            latest = int(0.95 * n_dates)
            delist_idx = int(local_rng.integers(earliest, latest))
            delist_date = dates[delist_idx]

            # Low-quality names suffer more severe delisting shocks.
            delist_shock = -local_rng.uniform(0.35, 0.90)
            clean_adj_close[delist_idx] *= (1 + delist_shock)

            # After delisting, there is no trading price.
            clean_adj_close[delist_idx + 1:] = np.nan

        frame = pd.DataFrame({
            "symbol": symbol,
            "timestamp": dates,
            "clean_adj_close": clean_adj_close,
            "market_return": market_returns,
            "quality_score": quality,
            "beta": beta,
            "annual_alpha": annual_alpha,
            "delisted": delisted,
            "delist_date": delist_date,
            "delist_shock": delist_shock
        })

        frames.append(frame)

        metadata_rows.append({
            "symbol": symbol,
            "quality_score": quality,
            "beta": beta,
            "annual_alpha": annual_alpha,
            "delisted": delisted,
            "delist_date": delist_date,
            "delist_shock": delist_shock
        })

    clean_panel = pd.concat(frames, ignore_index=True)
    asset_metadata = pd.DataFrame(metadata_rows)

    return clean_panel, asset_metadata

In [ ]:
config = SyntheticEquityConfig()
clean_panel, asset_metadata = simulate_clean_economic_prices(config)

clean_panel.head()

In [ ]:
asset_metadata.head()

## 4. Injecting corporate actions

We simulate stock splits and reverse splits.

A forward split reduces the raw price but does not change the economic value of holding the asset.

For example, in a 2-for-1 split:

- share count doubles;
- price per share roughly halves;
- total shareholder value is unchanged.

If raw prices are used directly, the split creates a fake negative return.

The adjusted close should remove this mechanical jump.

In [ ]:
def inject_corporate_actions(
    clean_panel: pd.DataFrame,
    config: SyntheticEquityConfig
) -> pd.DataFrame:
    """
    Create raw and adjusted prices by injecting split and reverse-split events.
    """
    local_rng = np.random.default_rng(config.seed + 1)
    panel = clean_panel.copy()

    panel["split_ratio"] = 1.0
    panel["corporate_action_type"] = "none"
    panel["cumulative_split_factor"] = 1.0

    symbols = panel["symbol"].unique()

    for symbol in symbols:
        symbol_idx = panel.index[panel["symbol"] == symbol].to_numpy()
        symbol_data = panel.loc[symbol_idx].copy()
        live_dates = symbol_data.loc[symbol_data["clean_adj_close"].notna(), "timestamp"]

        if len(live_dates) < 300:
            continue

        action_draw = local_rng.uniform()

        if action_draw < config.split_probability:
            action_type = "forward_split"
            ratio = float(local_rng.choice([2.0, 3.0, 4.0]))
        elif action_draw < config.split_probability + config.reverse_split_probability:
            action_type = "reverse_split"
            ratio = float(local_rng.choice([0.25, 0.5]))
        else:
            continue

        possible_dates = live_dates.iloc[int(0.25 * len(live_dates)): int(0.85 * len(live_dates))]

        if possible_dates.empty:
            continue

        action_date = pd.Timestamp(local_rng.choice(possible_dates.to_numpy()))

        action_mask = (panel["symbol"] == symbol) & (panel["timestamp"] >= action_date)
        action_day_mask = (panel["symbol"] == symbol) & (panel["timestamp"] == action_date)

        panel.loc[action_day_mask, "split_ratio"] = ratio
        panel.loc[action_day_mask, "corporate_action_type"] = action_type

        # cumulative split factor: new shares per old share
        panel.loc[action_mask, "cumulative_split_factor"] = ratio

    panel["raw_close"] = panel["clean_adj_close"] / panel["cumulative_split_factor"]
    panel["vendor_adj_close"] = panel["raw_close"] * panel["cumulative_split_factor"]

    return panel


corporate_panel = inject_corporate_actions(clean_panel, config)

corporate_panel[
    ["symbol", "timestamp", "raw_close", "vendor_adj_close", "split_ratio", "corporate_action_type"]
].head()

In [ ]:
corporate_actions = corporate_panel[corporate_panel["corporate_action_type"] != "none"][
    ["symbol", "timestamp", "split_ratio", "corporate_action_type"]
].copy()

corporate_actions.head()

## 5. Injecting missingness and bad ticks

We now create a vendor-like panel with realistic defects:

1. **random missing values**;
2. **block missing values**;
3. **single-point bad ticks**.

A single bad price creates two abnormal returns: one into the bad observation and one out of it.

In [ ]:
def inject_vendor_defects(
    panel: pd.DataFrame,
    config: SyntheticEquityConfig
) -> pd.DataFrame:
    """
    Inject missing values and bad ticks into raw and adjusted prices.
    """
    local_rng = np.random.default_rng(config.seed + 2)

    out = panel.copy()
    out["is_random_missing_injected"] = False
    out["is_block_missing_injected"] = False
    out["is_bad_tick_injected"] = False

    price_columns = ["raw_close", "vendor_adj_close"]

    # Random missingness among live observations.
    live_mask = out["vendor_adj_close"].notna()
    random_missing_mask = live_mask & (local_rng.uniform(size=len(out)) < config.random_missing_probability)

    out.loc[random_missing_mask, price_columns] = np.nan
    out.loc[random_missing_mask, "is_random_missing_injected"] = True

    # Block missingness by symbol.
    for symbol in out["symbol"].unique():
        if local_rng.uniform() > config.block_missing_probability:
            continue

        symbol_mask = out["symbol"] == symbol
        live_indices = out.index[symbol_mask & out["vendor_adj_close"].notna()].to_numpy()

        if len(live_indices) < 200:
            continue

        block_length = int(local_rng.integers(4, 18))
        start_pos = int(local_rng.integers(50, max(51, len(live_indices) - block_length)))
        block_indices = live_indices[start_pos: start_pos + block_length]

        out.loc[block_indices, price_columns] = np.nan
        out.loc[block_indices, "is_block_missing_injected"] = True

    # Bad ticks: multiply one live price by a large/small factor.
    for symbol in out["symbol"].unique():
        if local_rng.uniform() > config.outlier_probability:
            continue

        symbol_mask = out["symbol"] == symbol
        live_indices = out.index[symbol_mask & out["vendor_adj_close"].notna()].to_numpy()

        if len(live_indices) < 200:
            continue

        idx = int(local_rng.choice(live_indices[50:-50]))
        factor = float(local_rng.choice([0.08, 0.12, 5.0, 8.0, 12.0]))

        out.loc[idx, price_columns] = out.loc[idx, price_columns] * factor
        out.loc[idx, "is_bad_tick_injected"] = True

    # Synthetic volume.
    out["volume"] = np.nan

    live_price_mask = out["vendor_adj_close"].notna()
    out.loc[live_price_mask, "volume"] = np.exp(
        local_rng.normal(loc=13.0, scale=0.55, size=int(live_price_mask.sum()))
    )

    # Missing price generally means missing volume in this synthetic dataset.
    out.loc[out["vendor_adj_close"].isna(), "volume"] = np.nan

    return out


vendor_panel = inject_vendor_defects(corporate_panel, config)

vendor_panel.head()

## 6. Computing raw and adjusted returns

We compute returns using both:

1. raw close;
2. vendor adjusted close.

The difference is most visible around corporate action dates.

In [ ]:
def add_return_columns(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Add raw and adjusted simple/log return columns.
    """
    out = panel.sort_values(["symbol", "timestamp"]).copy()

    out["raw_simple_return"] = out.groupby("symbol")["raw_close"].pct_change(fill_method=None)
    out["adj_simple_return"] = out.groupby("symbol")["vendor_adj_close"].pct_change(fill_method=None)

    out["raw_log_return"] = np.log1p(out["raw_simple_return"])
    out["adj_log_return"] = np.log1p(out["adj_simple_return"])

    return out


return_panel = add_return_columns(vendor_panel)

return_panel[
    ["symbol", "timestamp", "raw_close", "vendor_adj_close", "raw_simple_return", "adj_simple_return"]
].head()

## 7. Corporate action diagnostic

A split should create a large raw price move but not a large adjusted return.

We inspect the first corporate action in the synthetic dataset.

In [ ]:
if not corporate_actions.empty:
    action = corporate_actions.iloc[0]
    action_symbol = action["symbol"]
    action_date = action["timestamp"]

    window = return_panel[
        (return_panel["symbol"] == action_symbol)
        & (return_panel["timestamp"] >= action_date - pd.Timedelta(days=15))
        & (return_panel["timestamp"] <= action_date + pd.Timedelta(days=15))
    ].copy()

    display(action)
    display(window[[
        "timestamp",
        "raw_close",
        "vendor_adj_close",
        "split_ratio",
        "corporate_action_type",
        "raw_simple_return",
        "adj_simple_return"
    ]])
else:
    print("No corporate action generated in this simulation run.")

In [ ]:
if not corporate_actions.empty:
    plt.figure(figsize=(10, 6))
    plt.plot(window["timestamp"], window["raw_close"], marker="o", label="raw_close")
    plt.plot(window["timestamp"], window["vendor_adj_close"], marker="o", label="vendor_adj_close")
    plt.axvline(action_date, linestyle="--", label="corporate action date")
    plt.title(f"Raw vs Adjusted Close Around Corporate Action: {action_symbol}")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

## 8. Missingness diagnostics

Missingness should be measured before imputation or deletion.

Useful questions:

1. How many observations are missing by symbol?
2. Are missing values isolated or clustered?
3. Do missing values occur before delisting?
4. Are missing values associated with low-quality or illiquid names?
5. Would filling missing values introduce look-ahead bias?

In [ ]:
def missingness_report(panel: pd.DataFrame, price_col: str = "vendor_adj_close") -> pd.DataFrame:
    """
    Compute missingness diagnostics by symbol.
    """
    rows = []

    for symbol, group in panel.groupby("symbol"):
        group = group.sort_values("timestamp")
        missing = group[price_col].isna()

        # Count missing runs.
        run_id = (missing != missing.shift()).cumsum()
        missing_runs = group[missing].groupby(run_id[missing]).size()

        rows.append({
            "symbol": symbol,
            "n_rows": len(group),
            "n_missing": int(missing.sum()),
            "missing_rate": float(missing.mean()),
            "max_missing_run": int(missing_runs.max()) if len(missing_runs) else 0,
            "delisted": bool(group["delisted"].iloc[0]),
            "delist_date": group["delist_date"].iloc[0]
        })

    return pd.DataFrame(rows).sort_values("missing_rate", ascending=False)


missing_report = missingness_report(return_panel)

missing_report.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(missing_report["missing_rate"], bins=30)
plt.title("Distribution of Missingness Rates by Symbol")
plt.xlabel("Missingness rate")
plt.ylabel("Number of symbols")
plt.show()

## 9. Robust outlier detection with MAD z-scores

A standard z-score can be distorted by the very outliers it is meant to detect.

A more robust alternative uses the median absolute deviation:

$$
\text{MAD} = \operatorname{median}(|x_i - \operatorname{median}(x)|)
$$
The robust z-score is:

$$
\begin{aligned}
z_i^{\text{robust}} &= 0.6745 \cdot \frac{x_i - \operatorname{median}(x)} {\text{MAD}}
\end{aligned}
$$
We apply this by symbol to adjusted log returns.

In [ ]:
def robust_zscore_by_symbol(
    panel: pd.DataFrame,
    return_col: str = "adj_log_return",
    z_col: str = "robust_z_adj_log_return"
) -> pd.DataFrame:
    """
    Compute robust MAD z-score by symbol.
    """
    out = panel.copy()
    out[z_col] = np.nan

    for symbol, group in out.groupby("symbol"):
        values = group[return_col].replace([np.inf, -np.inf], np.nan).dropna()

        if len(values) < 50:
            continue

        median = values.median()
        mad = np.median(np.abs(values - median))

        if mad == 0 or np.isnan(mad):
            continue

        z = 0.6745 * (group[return_col] - median) / mad
        out.loc[group.index, z_col] = z

    return out


diagnostic_panel = robust_zscore_by_symbol(return_panel)
diagnostic_panel[["symbol", "timestamp", "adj_log_return", "robust_z_adj_log_return"]].head()

## 10. Round-trip bad tick detection

A bad price often creates a pattern like:

$$
\text{large positive return followed by large negative return}
$$
or:

$$
\text{large negative return followed by large positive return}
$$
This is different from a genuine jump that may not immediately reverse.

We therefore flag observations where:

1. the return into time $t$ is extreme;
2. the return out of time $t$ is extreme;
3. the two returns have opposite signs.

This suggests the price at time $t$ itself may be bad.

In [ ]:
def flag_suspicious_observations(
    panel: pd.DataFrame,
    z_threshold: float = 10.0
) -> pd.DataFrame:
    """
    Create diagnostic flags for outliers, round-trip bad ticks, and corporate-action effects.
    """
    out = panel.sort_values(["symbol", "timestamp"]).copy()

    out["is_extreme_return"] = out["robust_z_adj_log_return"].abs() > z_threshold

    out["next_adj_log_return"] = out.groupby("symbol")["adj_log_return"].shift(-1)
    out["next_robust_z"] = out.groupby("symbol")["robust_z_adj_log_return"].shift(-1)

    opposite_sign = np.sign(out["adj_log_return"]) != np.sign(out["next_adj_log_return"])

    out["is_round_trip_bad_tick_candidate"] = (
        (out["robust_z_adj_log_return"].abs() > z_threshold)
        & (out["next_robust_z"].abs() > z_threshold)
        & opposite_sign
    )

    # Corporate action diagnostic: raw return is extreme on a split day, adjusted return is not.
    out["is_corporate_action_raw_jump"] = (
        (out["corporate_action_type"] != "none")
        & (out["raw_log_return"].abs() > 0.20)
        & (out["adj_log_return"].abs() < 0.20)
    )

    out["needs_price_quarantine"] = (
        out["is_round_trip_bad_tick_candidate"]
        | (
            out["is_extreme_return"]
            & ~out["is_corporate_action_raw_jump"]
            & ~out["delisted"]
        )
    )

    return out


flagged_panel = flag_suspicious_observations(diagnostic_panel, z_threshold=10.0)

flag_summary = pd.Series({
    "extreme_adjusted_returns": int(flagged_panel["is_extreme_return"].sum()),
    "round_trip_candidates": int(flagged_panel["is_round_trip_bad_tick_candidate"].sum()),
    "corporate_action_raw_jumps": int(flagged_panel["is_corporate_action_raw_jump"].sum()),
    "price_quarantine_flags": int(flagged_panel["needs_price_quarantine"].sum()),
    "injected_bad_ticks": int(flagged_panel["is_bad_tick_injected"].sum())
})

flag_summary

In [ ]:
flagged_panel[
    flagged_panel["needs_price_quarantine"]
][[
    "symbol",
    "timestamp",
    "vendor_adj_close",
    "adj_simple_return",
    "robust_z_adj_log_return",
    "next_adj_log_return",
    "next_robust_z",
    "is_bad_tick_injected",
    "needs_price_quarantine"
]].head(10)

## 11. Sanitising prices and returns

A conservative approach is to quarantine suspicious prices, not merely winsorise returns.

Why?

Because if the price at time $t$ is bad, both returns $t-1 \to t$ and $t \to t+1$ can be contaminated.

The workflow below:

1. creates a sanitized adjusted close;
2. sets quarantined prices to missing;
3. recomputes returns using `fill_method=None`;
4. keeps flags so downstream notebooks know what happened.

This is safer than silently clipping returns.

In [ ]:
def build_sanitized_return_panel(flagged: pd.DataFrame) -> pd.DataFrame:
    """
    Build sanitized adjusted close and return columns.

    The function does not fill missing values. It marks suspicious prices
    as missing and recomputes returns honestly.
    """
    out = flagged.sort_values(["symbol", "timestamp"]).copy()

    out["sanitized_adj_close"] = out["vendor_adj_close"]

    out.loc[out["needs_price_quarantine"], "sanitized_adj_close"] = np.nan

    out["sanitized_simple_return"] = (
        out.groupby("symbol")["sanitized_adj_close"].pct_change(fill_method=None)
    )

    out["sanitized_log_return"] = np.log1p(out["sanitized_simple_return"])

    out["return_is_usable"] = (
        out["sanitized_log_return"].notna()
        & np.isfinite(out["sanitized_log_return"])
    )

    out["data_quality_bucket"] = "clean"
    out.loc[out["vendor_adj_close"].isna(), "data_quality_bucket"] = "missing_price"
    out.loc[out["needs_price_quarantine"], "data_quality_bucket"] = "quarantined_price"
    out.loc[out["corporate_action_type"] != "none", "data_quality_bucket"] = "corporate_action_day"
    out.loc[out["delisted"] & (out["timestamp"] >= out["delist_date"]), "data_quality_bucket"] = "delisted_or_after"

    return out


sanitized_panel = build_sanitized_return_panel(flagged_panel)

sanitized_panel[
    ["symbol", "timestamp", "vendor_adj_close", "sanitized_adj_close", "sanitized_log_return", "data_quality_bucket"]
].head()

In [ ]:
quality_counts = sanitized_panel["data_quality_bucket"].value_counts()

quality_counts

## 12. Comparing original and sanitized return distributions

A useful sanity check is to compare return distributions before and after sanitization.

Sanitization should reduce obvious data-error tails, but it should not remove all genuine market tail risk.

There is a danger here:

> Over-cleaning data can be just as misleading as under-cleaning data.

In [ ]:
comparison_returns = pd.DataFrame({
    "raw_log_return": sanitized_panel["raw_log_return"],
    "adjusted_log_return": sanitized_panel["adj_log_return"],
    "sanitized_log_return": sanitized_panel["sanitized_log_return"]
})

comparison_returns.describe(percentiles=[0.001, 0.01, 0.05, 0.95, 0.99, 0.999])

In [ ]:
plt.figure(figsize=(10, 6))

for col in ["adjusted_log_return", "sanitized_log_return"]:
    data = comparison_returns[col].replace([np.inf, -np.inf], np.nan).dropna()
    clipped = data.clip(data.quantile(0.001), data.quantile(0.999))
    plt.hist(clipped, bins=80, density=True, alpha=0.5, label=col)

plt.title("Adjusted vs Sanitized Log Return Distribution")
plt.xlabel("Log return, clipped to 0.1% and 99.9% quantiles for display")
plt.ylabel("Density")
plt.legend()
plt.show()

## 13. Survivorship bias demonstration

We compare two equal-weight universe returns:

1. **Survivor-only universe**  
   Only assets that are still alive at the final date.

2. **Point-in-time universe**  
   Assets are included while they are alive, including delisted assets before and during delisting.

The survivor-only universe is biased because it removes assets that performed badly or disappeared.

In [ ]:
def compute_equal_weight_universe_returns(
    panel: pd.DataFrame,
    return_col: str,
    survivor_only: bool
) -> pd.DataFrame:
    """
    Compute equal-weight returns across either survivor-only or point-in-time universe.
    """
    work = panel.copy()

    final_date = work["timestamp"].max()

    if survivor_only:
        final_alive_symbols = work.loc[
            (work["timestamp"] == final_date)
            & work["sanitized_adj_close"].notna(),
            "symbol"
        ].unique()

        work = work[work["symbol"].isin(final_alive_symbols)].copy()
        universe_label = "survivor_only"
    else:
        universe_label = "point_in_time"

    # Use only returns with usable data.
    daily = (
        work[work["return_is_usable"]]
        .groupby("timestamp")[return_col]
        .mean()
        .rename("equal_weight_return")
        .reset_index()
    )

    daily["universe"] = universe_label
    daily["equity_curve"] = (1 + daily["equal_weight_return"]).cumprod()

    return daily


survivor_returns = compute_equal_weight_universe_returns(
    sanitized_panel,
    return_col="sanitized_simple_return",
    survivor_only=True
)

point_in_time_returns = compute_equal_weight_universe_returns(
    sanitized_panel,
    return_col="sanitized_simple_return",
    survivor_only=False
)

bias_comparison = pd.concat([survivor_returns, point_in_time_returns], ignore_index=True)

bias_comparison.head()

In [ ]:
plt.figure(figsize=(10, 6))

for label, group in bias_comparison.groupby("universe"):
    plt.plot(group["timestamp"], group["equity_curve"], label=label)

plt.title("Survivorship Bias Demonstration")
plt.xlabel("Date")
plt.ylabel("Equal-weight equity curve")
plt.legend()
plt.show()

In [ ]:
def performance_summary(daily_returns: pd.DataFrame) -> pd.Series:
    """
    Compute simple performance statistics.
    """
    returns = daily_returns["equal_weight_return"].dropna()
    equity = (1 + returns).cumprod()

    ann_return = equity.iloc[-1] ** (252 / len(returns)) - 1
    ann_vol = returns.std(ddof=1) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    max_drawdown = drawdown.min()

    return pd.Series({
        "annualized_return": ann_return,
        "annualized_volatility": ann_vol,
        "sharpe_zero_rf": sharpe,
        "max_drawdown": max_drawdown,
        "num_days": len(returns)
    })


performance_table = (
    bias_comparison
    .groupby("universe")
    .apply(performance_summary)
    .reset_index()
)

performance_table

## 14. Bias report

A bias report should be saved alongside the return panel.

It should summarise:

- number of assets;
- number of delisted assets;
- missingness;
- number of corporate actions;
- outlier flags;
- quarantined observations;
- survivor-only versus point-in-time performance difference.

In [ ]:
def build_bias_report(
    sanitized: pd.DataFrame,
    performance_table: pd.DataFrame
) -> dict:
    """
    Build a JSON-serialisable bias and sanitization report.
    """
    survivor_perf = performance_table[
        performance_table["universe"] == "survivor_only"
    ].iloc[0].to_dict()

    pit_perf = performance_table[
        performance_table["universe"] == "point_in_time"
    ].iloc[0].to_dict()

    report = {
        "schema_version": "return_sanitization_v1",
        "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "num_assets": int(sanitized["symbol"].nunique()),
        "num_rows": int(len(sanitized)),
        "num_corporate_action_rows": int((sanitized["corporate_action_type"] != "none").sum()),
        "num_delisted_assets": int(sanitized.groupby("symbol")["delisted"].first().sum()),
        "num_missing_vendor_prices": int(sanitized["vendor_adj_close"].isna().sum()),
        "num_quarantined_prices": int(sanitized["needs_price_quarantine"].sum()),
        "num_injected_bad_ticks_known_in_synthetic_data": int(sanitized["is_bad_tick_injected"].sum()),
        "num_usable_returns": int(sanitized["return_is_usable"].sum()),
        "survivor_only_annualized_return": float(survivor_perf["annualized_return"]),
        "point_in_time_annualized_return": float(pit_perf["annualized_return"]),
        "survivorship_return_overstatement": float(
            survivor_perf["annualized_return"] - pit_perf["annualized_return"]
        ),
        "survivor_only_sharpe_zero_rf": float(survivor_perf["sharpe_zero_rf"]),
        "point_in_time_sharpe_zero_rf": float(pit_perf["sharpe_zero_rf"]),
        "survivorship_sharpe_overstatement": float(
            survivor_perf["sharpe_zero_rf"] - pit_perf["sharpe_zero_rf"]
        )
    }

    return report


bias_report = build_bias_report(sanitized_panel, performance_table)

bias_report

## 15. Saving sanitized returns and audit report

The saved output includes both:

1. a clean return panel for downstream models;
2. an audit report explaining the data quality decisions.

A future alpha notebook should use the sanitized return panel rather than raw vendor returns.

In [ ]:
output_dir = Path("data/processed/return_sanitization_v1")
output_dir.mkdir(parents=True, exist_ok=True)

sanitized_output_columns = [
    "symbol",
    "timestamp",
    "raw_close",
    "vendor_adj_close",
    "sanitized_adj_close",
    "raw_simple_return",
    "adj_simple_return",
    "sanitized_simple_return",
    "raw_log_return",
    "adj_log_return",
    "sanitized_log_return",
    "volume",
    "corporate_action_type",
    "split_ratio",
    "cumulative_split_factor",
    "delisted",
    "delist_date",
    "delist_shock",
    "is_random_missing_injected",
    "is_block_missing_injected",
    "is_bad_tick_injected",
    "is_extreme_return",
    "is_round_trip_bad_tick_candidate",
    "is_corporate_action_raw_jump",
    "needs_price_quarantine",
    "return_is_usable",
    "data_quality_bucket"
]

sanitized_output = sanitized_panel[sanitized_output_columns].copy()

csv_path = output_dir / "synthetic_sanitized_return_panel.csv"
report_path = output_dir / "bias_report.json"
performance_path = output_dir / "survivorship_performance_comparison.csv"
missingness_path = output_dir / "missingness_report.csv"

sanitized_output.to_csv(csv_path, index=False)
performance_table.to_csv(performance_path, index=False)
missing_report.to_csv(missingness_path, index=False)
report_path.write_text(json.dumps(bias_report, indent=2))

csv_path, report_path, performance_path, missingness_path

## 16. Practical checklist for return sanitization

Before returns enter a model, check:

1. **Corporate actions**  
   Are splits, reverse splits, dividends, mergers, and symbol changes handled?

2. **Adjusted vs raw prices**  
   Are returns computed from the correct price field?

3. **Outliers**  
   Are extreme returns genuine events or vendor errors?

4. **Round-trip patterns**  
   Does a huge move immediately reverse, suggesting one bad price?

5. **Missingness**  
   Are gaps isolated, block-like, delisting-related, or vendor-related?

6. **Survivorship**  
   Does the universe include assets that existed at the time, including failures?

7. **Point-in-time safety**  
   Was the data available at the historical timestamp?

8. **Auditability**  
   Can every exclusion or quarantine be explained?

9. **Over-cleaning risk**  
   Are genuine crashes being incorrectly removed?

10. **Downstream compatibility**  
   Does the sanitized panel preserve flags for later notebooks?

## 17. Limitations

This notebook deliberately uses a simplified synthetic universe.

### 17.1 Corporate actions are simplified

Only splits and reverse splits are simulated.

Real data may include:

- cash dividends;
- stock dividends;
- mergers;
- spin-offs;
- rights issues;
- ticker changes;
- CUSIP/ISIN changes;
- special distributions;
- suspensions and relistings.

### 17.2 Delisting mechanics are simplified

The synthetic delisting model applies a final negative shock and then removes trading prices.

Real delisting returns can be difficult to obtain and may depend on the delisting reason, exchange, bankruptcy process, cash settlement, merger terms, or OTC trading after delisting.

### 17.3 Outlier detection is not truth detection

A robust z-score can identify suspicious returns, but it cannot determine whether an event is economically real.

A genuine crash, takeover, limit move, or earnings surprise may look like an outlier.

Outlier flags should trigger review or rules-based quarantine, not automatic deletion.

### 17.4 Missingness is not random

The synthetic missingness process is partly random.

Real missingness is often informative. It may indicate illiquidity, suspension, exchange holidays, data vendor failures, or delisting.

### 17.5 Survivor-free data requires historical membership

A proper survivor-free backtest needs historical universe membership and identifiers.

It is not enough to download the current constituents of an index and run a backtest backwards.

### 17.6 Sanitization can change strategy performance

Cleaning data is not neutral.

Every flagging, filling, or exclusion rule can affect returns, risk, turnover, and signal behaviour.

That is why the rules must be explicit and version-controlled.

## 18. Research frontier and current directions

Return sanitization is increasingly treated as part of model risk management, not just data cleaning.

### 18.1 Point-in-time data and bitemporal storage

A robust research platform should distinguish:

- event time: when something happened;
- availability time: when the researcher could have known it;
- revision time: when a vendor changed the record.

This is especially important for fundamentals, macroeconomic data, index membership, corporate actions, and delisting information.

A future notebook could simulate revised macro data and show how using final revised values creates look-ahead bias.

### 18.2 Data quality observability

Modern data platforms monitor quality metrics continuously:

- row counts;
- missingness;
- duplicate keys;
- outlier rates;
- schema changes;
- stale prices;
- corporate-action frequency;
- distribution shifts.

A future notebook could build a data quality dashboard that tracks these metrics over time and alerts when they move outside expected ranges.

### 18.3 Vendor reconciliation

Institutional systems often compare multiple vendors.

If one vendor shows a 90% crash and another does not, the pipeline should flag the disagreement.

A future notebook could simulate two vendors with controlled inconsistencies and build a reconciliation layer.

### 18.4 Robust statistics and heavy tails

Financial returns genuinely have fat tails, so simple outlier deletion is dangerous.

Robust estimators, heavy-tailed models, and stress-testing frameworks are often better than pretending extreme returns are always errors.

A future notebook could compare Gaussian, Student-t, Laplace, and robust-M-estimator treatments of return tails.

### 18.5 Survivor-free universe reconstruction

A major source of bias is using only current index constituents.

A proper historical universe needs:

- historical membership;
- delisted securities;
- identifier mapping;
- corporate action history;
- availability timestamps.

A future notebook could reconstruct a synthetic index membership history and compare current-constituent backtests against point-in-time membership backtests.

### 18.6 ML-based anomaly detection

Machine learning can help detect suspicious observations, but it must be used carefully.

Anomaly detection models can learn market stress as if it were an error, or learn vendor-specific quirks that do not generalise.

A future notebook could compare robust z-scores, isolation forests, and autoencoder-based anomaly detection on synthetic data with known injected defects.

## 19. Suggested follow-up notebooks

This notebook naturally leads to:

1. **01_05_empirical_distribution_analyzer**  
   Studying return distributions after sanitization.

2. **01_08_data_leakage_detection_unit_tests**  
   Building tests that catch look-ahead and survivorship bias.

3. **03_14_information_coefficient_and_feature_decay**  
   Measuring alpha quality on a sanitized return panel.

4. **04_06_var_cvar_and_stress_testing**  
   Using cleaned returns without deleting genuine tail risk.

5. **05_01_vectorized_backtest_engine**  
   Feeding sanitized, point-in-time returns into the backtester.

6. **05_07_purged_kfold_and_embargo_cv**  
   Preventing leakage in financial machine learning.

7. **07_01_multi_asset_cta_research_pipeline**  
   Using the sanitized panel as part of an end-to-end research pipeline.

## 20. Summary

This notebook built a synthetic but realistic return sanitization pipeline.

It demonstrated how four common problems can distort research:

1. bad ticks and outliers;
2. corporate actions;
3. survivorship bias;
4. missing data.

The main outputs were:

- a sanitized adjusted price series;
- a sanitized return panel;
- missingness diagnostics;
- outlier flags;
- corporate-action diagnostics;
- survivor-only versus point-in-time performance comparison;
- a saved bias report.

The key computational takeaway is:

> Return data should not be treated as a direct output of prices. It should be the result of a documented sanitization and validation pipeline.

The key financial takeaway is:

> Fake alpha often comes from bad data, biased universes, or silent cleaning assumptions. Bias control is therefore part of the model, not admin work.

## 21. Further reading

### Corporate actions and adjusted prices

- SEC investor education material on stock splits and reverse stock splits.
- Exchange corporate action feeds and vendor documentation.
- Yahoo Finance documentation on adjusted close methodology.

### Survivorship and point-in-time data

- CRSP survivor-bias-free database documentation.
- Academic literature on survivorship bias and look-ahead benchmark bias.
- Index membership reconstruction and historical constituent datasets.

### Robust return cleaning

- Robust statistics and median absolute deviation.
- Hampel filters for time-series outliers.
- Heavy-tailed return modelling.
- Data quality observability systems.

### Future extensions

- Bitemporal data stores.
- Vendor reconciliation.
- ML anomaly detection.
- Delisting return modelling.
- Corporate-action-aware identifier mapping.